In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# Nature-style font settings
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.dpi': 300,
})

disease_list = ['AID', 'AKI', 'FH', 'MACE', 'T2DM']
reviewers = ['AA', 'BB']
model_colors = {'ALL': '#C0392B', 'TEXT': '#2980B9', 'DIAG': '#27AE60'}
model_labels = {'ALL': 'All information', 'TEXT': 'Text only', 'DIAG': 'Diagram only'}

In [ ]:
# Load all data, compute mean across reviewers for each disease
disease_data = {}
for disease in disease_list:
    dfs = [pd.read_csv(f'{disease}_{r}.csv').set_index('Setting')
           for r in reviewers]
    disease_data[disease] = (dfs[0] + dfs[1]) / 2  # mean across 2 reviewers

# Also compute overall mean across all diseases
overall_data = sum(disease_data.values()) / len(disease_list)

In [12]:
def draw_radar(df, title, save_path):
    categories = list(df.columns)
    N = len(categories)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    for model in df.index:
        values = df.loc[model].tolist() + [df.loc[model].tolist()[0]]
        ax.plot(angles, values, label=model_labels[model],
                linewidth=2.5, color=model_colors[model])
        ax.fill(angles, values, alpha=0.15, color=model_colors[model])

    # Axis labels with alignment per quadrant
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=13)
    for label, angle in zip(ax.get_xticklabels(), angles[:-1]):
        angle_deg = np.degrees(angle)
        if 90 < angle_deg <= 270:
            label.set_horizontalalignment('right')
        else:
            label.set_horizontalalignment('left')

    ax.set_rlabel_position(22.5)
    ax.set_yticks([1, 2, 3, 4])
    ax.set_yticklabels(['1', '2', '3', '4'], color='grey', size=11)
    ax.set_ylim(0, 4)

    # Minimal grid style (Nature)
    ax.spines['polar'].set_visible(False)
    ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.7)

    ax.set_title(title, size=16, fontweight='bold', pad=20)
    # No legend — saved separately as fig/legend.png

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', format='png')
    plt.show()
    print(f'Saved: {save_path}')

In [ ]:
import os
import matplotlib.patches as mpatches
os.makedirs('fig', exist_ok=True)

disease_full_names = {
    'AID': 'Autoimmune Disease',
    'AKI': 'Acute Kidney Injury',
    'FH': 'Familial Hypercholesterolemia',
    'MACE': 'Major Adverse Cardiac Events',
    'T2DM': 'Type 2 Diabetes Mellitus',
}

# Per-disease plots (no legend)
for disease in disease_list:
    draw_radar(disease_data[disease], disease_full_names[disease],
               f'fig/radar_{disease}.png')

# Overall plot (no legend)
draw_radar(overall_data, 'ChatGPT o3', 'fig/radar_overall.png')

# Standalone legend PNG
fig_leg, ax_leg = plt.subplots(figsize=(3, 1.2))
ax_leg.axis('off')
handles = [mpatches.Patch(color=model_colors[k], label=model_labels[k])
           for k in ['ALL', 'TEXT', 'DIAG']]
ax_leg.legend(handles=handles, loc='center', frameon=False, fontsize=13, ncol=1, handleheight=0.21)
plt.tight_layout()
plt.savefig('fig/legend.png', dpi=300, bbox_inches='tight', format='png')
plt.show()
print('Saved: fig/legend.png')